In [ ]:
from pymilvus import MilvusClient, DataType, Function, FunctionType

# 云端向量数据库
CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
TOKEN = "YOUR_MILVUS_TOKEN"


# CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-T5-BM25-L2
# TOKEN = "YOUR_MILVUS_TOKEN"
# CLUSTER_ENDPOINT = "YOUR_MILVUS_ENDPOINT" # Serverless-Not-Java
# TOKEN = "YOUR_MILVUS_TOKEN"

client = MilvusClient(
    uri=CLUSTER_ENDPOINT,
    token=TOKEN 
)

client.list_collections()

## RAG案例

In [ ]:
import pandas as pd
from tqdm import tqdm

group_id = 18
# prefix = 2 # 50repo前缀为1，mergebert前缀为2 
repo_name = "mergebert_80"
local_file_path = 'dataset/mergebert/vector/' + repo_name + '_Conflicts.txt'

# 使用 pandas 读取本地文件
pair = pd.read_csv(local_file_path, sep='\t', header=None).drop_duplicates()
conflicts = pair[0].values
res_labels = pair[1].values
global_id = pair[2].values

data_pairs = list(zip(conflicts, res_labels, global_id))

# 打印前几行，查看数据是否正确读取
print(conflicts.size) 
print(conflicts[:3])
print(res_labels[:3]) 
print(global_id[:3])

In [ ]:
client.drop_collection(collection_name=repo_name)

schema = MilvusClient.create_schema(
    auto_id=True,
    enable_dynamic_field=True,
 )
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="global_id", datatype=DataType.INT32)
schema.add_field(field_name="chunk_content", datatype=DataType.VARCHAR, max_length=2048, enable_analyzer=True)
schema.add_field(field_name="chunk_resolution", datatype=DataType.VARCHAR, max_length=1024)
schema.add_field(field_name="chunk_sparse_embedding", datatype=DataType.SPARSE_FLOAT_VECTOR)
schema.add_field(field_name="chunk_embedding", datatype=DataType.FLOAT_VECTOR, dim=768)

# BM25函数
bm25_function = Function(
    name="text_bm25_emb",
    input_field_names="chunk_content",
    output_field_names="chunk_sparse_embedding",
    function_type=FunctionType.BM25,
 )
schema.add_function(bm25_function)

index_params = MilvusClient.prepare_index_params()
index_params.add_index(
    field_name="chunk_embedding",
    metric_type="L2",
    index_type="AUTOINDEX",
    index_name="vector_index"
 )
index_params.add_index(
    field_name="chunk_sparse_embedding",
    index_name="sparse_inverted_index",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
    params={"drop_ratio_build": 0.2},
 )


client.create_collection(
    collection_name=repo_name, 
    schema=schema, 
    index_params=index_params
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import numpy as np
from tqdm import tqdm

# 加载本地 CodeT5 模型
local_model_path = "./codeT5/codet5-base"
def download_or_load_model(local_model_path):
    try:
        tokenizer = AutoTokenizer.from_pretrained(local_model_path) # 尝试从本地加载模型
        model = AutoModelForSeq2SeqLM.from_pretrained(local_model_path).to("cuda" if torch.cuda.is_available() else "cpu")
        print("使用本地模型")
    except Exception as e:
        print(f"从本地加载模型失败，错误：{e}")
        tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base") # 本地不存在则从网络下载
        model = AutoModelForSeq2SeqLM.from_pretrained("Salesforce/codet5-base").to("cuda" if torch.cuda.is_available() else "cpu")
        # 保存到本地
        tokenizer.save_pretrained(local_model_path)
        model.save_pretrained(local_model_path)
        print("从网络下载并保存本地模型")
    return tokenizer, model
tokenizer, model = download_or_load_model(local_model_path)

"""向量数据库的数据插入"""
for conflict, resolution, g_id in tqdm(data_pairs, desc="Processing conflicts and resolutions"):

    inputs = tokenizer(conflict, padding=True, truncation=True, return_tensors="pt").to(model.device)
    # 使用 CodeT5 获取嵌入
    with torch.no_grad():
        encoder_outputs = model.encoder(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
        embeddings = encoder_outputs.last_hidden_state.mean(dim=1).cpu().numpy()  # 获取句子级别的 embedding
    encode = embeddings[0]

    if not isinstance(resolution, str) or not resolution.strip(): # 处理 resolution 字段
        res_data = "_newline_"
    else:
        res_data = str(resolution).strip()

    res = client.insert(
        collection_name=repo_name,
        data=[
            {
                "group_id": group_id,
                "global_id": g_id,
                "chunk_content": conflict.encode('utf-8')[:2048].decode('utf-8', errors='ignore'),
                "chunk_resolution": res_data.encode('utf-8')[:1024].decode('utf-8', errors='ignore'),
                "chunk_embedding": encode
            }
        ]
    )

In [ ]:
import matplotlib.pyplot as plt

with open(local_file_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

part1_lengths = []
part2_lengths = []

for line in tqdm(lines):
    # 使用 \t 分割每行
    parts = line.strip().split('\t', 1)
    
    if len(parts) == 2:  
        part1_length = len(parts[0].strip())  # 计算第一部分的字符长度
        part2_length = len(parts[1].strip())  # 计算第二部分的字符长度
        
        part1_lengths.append(part1_length)
        part2_lengths.append(part2_length)

plt.figure(figsize=(14, 6))

# 第一部分
plt.subplot(1, 2, 1)
plt.hist(part1_lengths, bins=100, color='#3498db', edgecolor='black')  # 使用蓝色
plt.title("Character Length Distribution of Conflict")
plt.xlabel("Number of Characters")
plt.ylabel("Frequency")
plt.xlim(0, 4000)  # 根据实际需要调整x轴范围
plt.grid(True)

# 第二部分
plt.subplot(1, 2, 2)
plt.hist(part2_lengths, bins=100, color='#e74c3c', edgecolor='black')  # 使用红色
plt.title("Character Length Distribution of Resolution")
plt.xlabel("Number of Characters")
plt.ylabel("Frequency")
plt.xlim(0, 2000)  # 根据实际需要调整x轴范围
plt.grid(True)

plt.tight_layout()  # 自动调整子图参数，使之填充整个图像区域
plt.show()

### 开始使用

In [ ]:
from database_api_T5_BM25_L2 import query_similar
query = """concurrent_change _newline_         if (!(cmd instanceof ListVolumesCmdByAdmin)) { _newline_  _newline_ base_content _newline_         if(!isRootAdmin){ _newline_  _newline_ incoming_change _newline_  _newline_        """

global_ids, related_cs, related_rs, timestamps = query_similar(CLUSTER_ENDPOINT, TOKEN, repo_name, query, 3)

# 输出结果
print("Id:", global_ids)
print("conflict:", related_cs)
print("resolution:", related_rs)
print("timestamp:", timestamps)